In [1]:
from langchain_community.retrievers import WikipediaRetriever

Wikipedia Retriever

In [2]:
#
# %pip install wikipedia

In [3]:
# Initialize the retriever (optional: set language and top_k)
retriever = WikipediaRetriever(top_k_results=2, lang="en")

In [8]:
import wikipedia
import wikipedia.wikipedia as wiki_core

# Set language
wikipedia.set_lang("en")

# Set proper User-Agent
# Replace email with your real email if possible
wikipedia.set_user_agent(
    "RAG-Learning-Notebook/1.0 (balramt02@gmsail.com)"
)

# Force HTTPS API URL
wiki_core.API_URL = "https://en.wikipedia.org/w/api.php"

In [9]:
page = wikipedia.page("India–Pakistan relations", auto_suggest=False)

print(page.title)
print(page.summary[:1000])

India–Pakistan relations
India and Pakistan have a complex and largely hostile relationship that is rooted in a multitude of historical and political events, most notably the partition of British India in August 1947.
Two years after World War II, the United Kingdom formally dissolved British India, dividing it into two new sovereign nations: the Union of India and Pakistan. The partitioning of the former British colony resulted in the displacement of up to 15 million people, with the death toll estimated to have reached between several hundred thousand and one million people as Hindus and Muslims migrated in opposite directions across the Radcliffe Line to reach India and Pakistan, respectively. In 1950, India emerged as a secular republic with a Hindu-majority population. Shortly afterwards, in 1956, Pakistan emerged as an Islamic republic with a Muslim-majority population.
While the two South Asian countries established full diplomatic ties shortly after their formal independence, t

In [13]:

# Define your query
query = "the geopolitical history of india and pakistan from the perspective of a chinese"

# Get relevant Wikipedia documents
docs = retriever.invoke(query)

In [14]:
docs

[Document(metadata={'title': 'India–Pakistan war of 1971', 'summary': "The India–Pakistan war of 1971, also known as the third Indo-Pakistani war, was a military confrontation between India and Pakistan that occurred during the Bangladesh Liberation War in East Pakistan from 3 December 1971 until the Pakistani capitulation in Dhaka on 16 December 1971.  The war began with Pakistan's Operation Chengiz Khan, consisting of preemptive aerial strikes on eight Indian air stations. The strikes led to India declaring war on Pakistan, marking their entry into the war for East Pakistan's independence, on the side of Bengali nationalist forces. India's entry expanded the existing conflict with Indian and Pakistani forces engaging on both the eastern and western fronts.\nThirteen days after the war started, India achieved a clear upper hand, and the Eastern Command of the Pakistan military signed the instrument of surrender on 16 December 1971 in Dhaka, marking the formation of East Pakistan as th

In [15]:
# Print retrieved content
for i, doc in enumerate(docs):
    print(f"\n--- Result {i+1} ---")
    print(f"Content:\n{doc.page_content}...")  # truncate for display


--- Result 1 ---
Content:
The India–Pakistan war of 1971, also known as the third Indo-Pakistani war, was a military confrontation between India and Pakistan that occurred during the Bangladesh Liberation War in East Pakistan from 3 December 1971 until the Pakistani capitulation in Dhaka on 16 December 1971.  The war began with Pakistan's Operation Chengiz Khan, consisting of preemptive aerial strikes on eight Indian air stations. The strikes led to India declaring war on Pakistan, marking their entry into the war for East Pakistan's independence, on the side of Bengali nationalist forces. India's entry expanded the existing conflict with Indian and Pakistani forces engaging on both the eastern and western fronts.
Thirteen days after the war started, India achieved a clear upper hand, and the Eastern Command of the Pakistan military signed the instrument of surrender on 16 December 1971 in Dhaka, marking the formation of East Pakistan as the new nation of Bangladesh. Approximately 93,

Vector Store Retriever

In [2]:
from langchain_community.vectorstores import Chroma
#from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings
from langchain_core.documents import Document 

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [14]:
# Step 1: Your source documents
documents = [
    Document(page_content="LangChain helps developers build LLM applications easily."),
    Document(page_content="Chroma is a vector database optimized for LLM-based search."),
    Document(page_content="Embeddings convert text into high-dimensional vectors."),
    Document(page_content="OpenAI provides powerful embedding models."),
]

In [19]:
# Step 2: Initialize embedding model
embedding_model = "nomic-embed-text"
embeddings = OllamaEmbeddings(model = embedding_model)

# Step 3: Create Chroma vector store in memory
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    collection_name="my_collection"
)

In [20]:
# Step 4: Convert vectorstore into a retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

In [21]:
query = "What is Chroma used for?"
results = retriever.invoke(query)

In [22]:
for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
Chroma is a vector database optimized for LLM-based search.

--- Result 2 ---
LangChain helps developers build LLM applications easily.


In [ ]:
results = vectorstore.similarity_search(query, k=2)  

In [24]:
for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
Chroma is a vector database optimized for LLM-based search.

--- Result 2 ---
LangChain helps developers build LLM applications easily.


# Here we are getting same result as above by using vectore storers only, so why we need to to the same taks from retrievers is because retrievers can do more extra and modified and advanced retrieves strategy that is not achieved via vectore stores ok.


Maximal Marginal Relevance(MMR)

In [3]:
# Sample documents
docs = [
    Document(page_content="LangChain makes it easy to work with LLMs."),
    Document(page_content="LangChain is used to build LLM based applications."),
    Document(page_content="Chroma is used to store and search document embeddings."),
    Document(page_content="Embeddings are vector representations of text."),
    Document(page_content="MMR helps you get diverse results when doing similarity search."),
    Document(page_content="LangChain supports Chroma, FAISS, Pinecone, and more."),
]

In [4]:
from langchain_community.vectorstores import FAISS

# Step 2: Initialize embedding model
embedding_model = "nomic-embed-text"
embeddings = OllamaEmbeddings(model = embedding_model)

# Step 3 Create the FAISS storwe form documents

vectorstore = FAISS.from_documents(
    documents= docs,
    embedding=embeddings
)

In [8]:
# Enable MMR in the retriever
retriever = vectorstore.as_retriever(
    search_type="mmr",                   # <-- This enables MMR
    search_kwargs={"k": 3, "lambda_mult": 1}  # k = top results, lambda_mult = relevance-diversity balance
)

In [9]:
query = "What is langchain?"
results = retriever.invoke(query)

In [10]:
for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
LangChain is used to build LLM based applications.

--- Result 2 ---
Chroma is used to store and search document embeddings.

--- Result 3 ---
LangChain makes it easy to work with LLMs.


Multiquery Retriever

In [1]:
pip install langchain-ollama


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
!ollama list

]11;?\NAME                       ID              SIZE      MODIFIED   
nomic-embed-text:latest    0a109f422b47    274 MB    4 days ago    
llama3.2:1b                baf6a787fdff    1.3 GB    4 days ago    


In [7]:
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_classic.retrievers.multi_query import MultiQueryRetriever

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [9]:
all_docs = [
    Document(
        page_content="Regular walking boosts heart health and can reduce symptoms of depression.",
        metadata={"source": "H1"}
    ),
    Document(
        page_content="Consuming leafy greens and fruits helps detox the body and improve longevity.",
        metadata={"source": "H2"}
    ),
    Document(
        page_content="Deep sleep is crucial for cellular repair and emotional regulation.",
        metadata={"source": "H3"}
    ),
    Document(
        page_content="Mindfulness and controlled breathing lower cortisol and improve mental clarity.",
        metadata={"source": "H4"}
    ),
    Document(
        page_content="Drinking sufficient water throughout the day helps maintain metabolism and energy.",
        metadata={"source": "H5"}
    ),
    Document(
        page_content="The solar energy system in modern homes helps balance electricity demand.",
        metadata={"source": "I1"}
    ),
    Document(
        page_content="Python balances readability with power, making it a popular system design language.",
        metadata={"source": "I2"}
    ),
    Document(
        page_content="Photosynthesis enables plants to produce energy by converting sunlight.",
        metadata={"source": "I3"}
    ),
    Document(
        page_content="The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.",
        metadata={"source": "I4"}
    ),
    Document(
        page_content="Black holes bend spacetime and store immense gravitational energy.",
        metadata={"source": "I5"}
    ),
]

In [10]:
embedding_model = OllamaEmbeddings(
    model="nomic-embed-text"
)

In [ ]:
# Create a FAISS vector store from the documents
vectorstore = FAISS.from_documents(
    documents=all_docs,
    embedding=embedding_model
)

In [12]:
similarity_retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

In [13]:
llm = ChatOllama(
    model="llama3.2:1b",
    temperature=0
)

In [14]:
multiquery_retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(search_kwargs={"k": 5}),
    llm=llm
)

In [16]:
query = "How to improve energy levels and maintain balance?"

similarity_results = similarity_retriever.invoke(query)
multiquery_results = multiquery_retriever.invoke(query)

In [17]:
print("SIMILARITY RETRIEVER RESULTS")
print("=" * 80)

for i, doc in enumerate(similarity_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)
    print("Metadata:", doc.metadata)

SIMILARITY RETRIEVER RESULTS

--- Result 1 ---
Drinking sufficient water throughout the day helps maintain metabolism and energy.
Metadata: {'source': 'H5'}

--- Result 2 ---
The solar energy system in modern homes helps balance electricity demand.
Metadata: {'source': 'I1'}

--- Result 3 ---
Mindfulness and controlled breathing lower cortisol and improve mental clarity.
Metadata: {'source': 'H4'}

--- Result 4 ---
Consuming leafy greens and fruits helps detox the body and improve longevity.
Metadata: {'source': 'H2'}

--- Result 5 ---
Photosynthesis enables plants to produce energy by converting sunlight.
Metadata: {'source': 'I3'}


In [18]:
print("\n" + "*" * 150 + "\n")

print("MULTI QUERY RETRIEVER RESULTS")
print("=" * 80)

for i, doc in enumerate(multiquery_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)
    print("Metadata:", doc.metadata)


******************************************************************************************************************************************************

MULTI QUERY RETRIEVER RESULTS

--- Result 1 ---
The solar energy system in modern homes helps balance electricity demand.
Metadata: {'source': 'I1'}

--- Result 2 ---
Drinking sufficient water throughout the day helps maintain metabolism and energy.
Metadata: {'source': 'H5'}

--- Result 3 ---
Photosynthesis enables plants to produce energy by converting sunlight.
Metadata: {'source': 'I3'}

--- Result 4 ---
Consuming leafy greens and fruits helps detox the body and improve longevity.
Metadata: {'source': 'H2'}

--- Result 5 ---
Mindfulness and controlled breathing lower cortisol and improve mental clarity.
Metadata: {'source': 'H4'}

--- Result 6 ---
Regular walking boosts heart health and can reduce symptoms of depression.
Metadata: {'source': 'H1'}

--- Result 7 ---
Deep sleep is crucial for cellular repair and emotional regulation.

## ContextualCompressionRetriever

In [32]:
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

In [33]:
# Recreate the document objects from the previous data
docs = [
    Document(page_content=(
        """The Grand Canyon is one of the most visited natural wonders in the world.
        Photosynthesis is the process by which green plants convert sunlight into energy.
        Millions of tourists travel to see it every year. The rocks date back millions of years."""
    ), metadata={"source": "Doc1"}),

    Document(page_content=(
        """In medieval Europe, castles were built primarily for defense.
        The chlorophyll in plant cells captures sunlight during photosynthesis.
        Knights wore armor made of metal. Siege weapons were often used to breach castle walls."""
    ), metadata={"source": "Doc2"}),

    Document(page_content=(
        """Basketball was invented by Dr. James Naismith in the late 19th century.
        It was originally played with a soccer ball and peach baskets. NBA is now a global league."""
    ), metadata={"source": "Doc3"}),

    Document(page_content=(
        """The history of cinema began in the late 1800s. Silent films were the earliest form.
        Thomas Edison was among the pioneers. Photosynthesis does not occur in animal cells.
        Modern filmmaking involves complex CGI and sound design."""
    ), metadata={"source": "Doc4"})
]

In [34]:
embedding_model = OllamaEmbeddings(
    model="nomic-embed-text"
)

In [35]:
# Create a FAISS vector store from the documents
vectorstore = FAISS.from_documents(
    documents=all_docs,
    embedding=embedding_model
)

In [36]:
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

In [37]:
# Set up the compressor using an LLM
llm = ChatOllama(
    model="llama3.2:1b",
    temperature=0
)
compressor = LLMChainExtractor.from_llm(llm)

In [38]:
# Create the contextual compression retriever
compression_retriever = ContextualCompressionRetriever(
    base_retriever=base_retriever,
    base_compressor=compressor
)

In [39]:
# Query the retriever
query = "What is photosynthesis?"
compressed_results = compression_retriever.invoke(query)

In [40]:
for i, doc in enumerate(compressed_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)
